# E5e γ-β — reasoning-mode amplification (qwen3-vl-8b instruct vs thinking)

Top-to-bottom reproducer for `docs/insights/E5e-mathvista-reasoning-evidence.md`.

Source: `outputs/experiment_e5e_mathvista_reasoning/qwen3-vl-8b-{instruct,thinking}/20260428-114421/` (post-C-form reaggregate).
Per-cell artefact: `docs/insights/_data/experiment_e5e_mathvista_reasoning_per_cell.csv`.

## Headline

Reasoning amplifies anchor pull most strongly on **correct-base** records (df ratio ×12.7),
where instruct mode shows almost none (df = 0.021). The H2 wrong > correct asymmetry that
holds across the main panel collapses in thinking mode — direct evidence that the H7
continuous-confidence monotonicity from `L1-confidence-modulation-evidence.md` breaks
down with chain-of-thought.

In [ ]:
import sys
from pathlib import Path
import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT / 'scripts'))
pd.set_option('display.precision', 3)

## 1. Build / load per-cell summary

`scripts/analyze_e5e_wrong_correct.py` walks `outputs/experiment_e5e_mathvista_reasoning/<model>/<run>/predictions.jsonl`
and emits `docs/insights/_data/experiment_e5e_mathvista_reasoning_per_cell.csv` with rows split by
(model, condition, S-stratum, base_correct).

In [ ]:
import subprocess
subprocess.run(['uv', 'run', 'python', str(ROOT / 'scripts' / 'analyze_e5e_wrong_correct.py'),
                '--exp-dir', 'experiment_e5e_mathvista_reasoning'], cwd=ROOT, check=True)
per_cell = pd.read_csv(ROOT / 'docs' / 'insights' / '_data' / 'experiment_e5e_mathvista_reasoning_per_cell.csv')
per_cell

## 2. Wrong-base / correct-base split (the new finding)

All-base ratio is ×2.9 on df. The correct-base cell hides a ×12.7 amplification.

In [ ]:
anchor = per_cell[per_cell['cond_class'] == 'a'].copy()
anchor['base'] = anchor['base_correct'].map({True: 'correct', False: 'wrong'})
df_pivot = anchor.pivot_table(index='base', columns='model', values='direction_follow_M2', aggfunc='first').round(3)
ad_pivot = anchor.pivot_table(index='base', columns='model', values='adopt_M2', aggfunc='first').round(3)
print('=== df(a) C-form ===')
print(df_pivot)
print()
print('=== adopt(a) ===')
print(ad_pivot)
print()
print('=== thinking / instruct df(a) ratio ===')
ratio = df_pivot['qwen3-vl-8b-thinking'] / df_pivot['qwen3-vl-8b-instruct']
print(ratio.round(2))

## 3. Masked-arm contrast (digit-pixel causality)

(a − m) gap on `df` and `adopt`, wrong-base then correct-base.

In [ ]:
anchor_m = per_cell[per_cell['cond_class'].isin(['a', 'm'])].copy()
anchor_m['base'] = anchor_m['base_correct'].map({True: 'correct', False: 'wrong'})
wide = anchor_m.pivot_table(
    index=['model', 'base'], columns='cond_class',
    values=['direction_follow_M2', 'adopt_M2'], aggfunc='first',
)
wide.columns = [f'{m}_{c}' for m, c in wide.columns]
wide['df_a_minus_m'] = wide['direction_follow_M2_a'] - wide['direction_follow_M2_m']
wide['adopt_a_minus_m'] = wide['adopt_M2_a'] - wide['adopt_M2_m']
print(wide.round(3).to_string())

## 4. All-base headline (matches `E5e-mathvista-evidence.md §6`)

Read straight from `summary.json` for both models. Verifies the all-base ×1.6 / ×2.9 ratios.

In [ ]:
import json
rows = []
for model in ['qwen3-vl-8b-instruct', 'qwen3-vl-8b-thinking']:
    p = ROOT / 'outputs' / 'experiment_e5e_mathvista_reasoning' / model / '20260428-114421' / 'summary.json'
    summary = json.load(open(p))
    for cond_label, cond_key in [('b', 'target_only'),
                                  ('a-S1', 'target_plus_irrelevant_number_S1'),
                                  ('m-S1', 'target_plus_irrelevant_number_masked_S1'),
                                  ('d', 'target_plus_irrelevant_neutral')]:
        s = summary[cond_key]
        rows.append({
            'model': model, 'cond': cond_label, 'n': s['count'],
            'em': s.get('accuracy_exact'),
            'adopt': s.get('anchor_adoption_rate'),
            'df': s.get('anchor_direction_follow_rate'),
        })
headline = pd.DataFrame(rows).round(3)
print(headline.to_string(index=False))

## 5. Cross-link to the rest of the paper

- **§1** opening hook — reasoning amplifies, not suppresses, against the DPT prior.
- **§6** confidence modulation — H7 monotonicity holds for instruct + main panel; **collapses** in thinking.
  See `docs/insights/L1-confidence-modulation-evidence.md` for the panel-side numbers (Q4−Q1 +0.152).
- **§8** future work F3 — single-cell stake; multi-dataset extension proposed in evidence §5.